<a href="https://colab.research.google.com/github/Medypan/fintech545_medypan/blob/main/545Assignment2MedyPan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import numpy as np
import pandas as pd

# DEFINE missing_cov
def missing_cov(x, skipMiss=True, fun=np.cov):
    """
    x: numpy array with NaN for missing
    skipMiss = True -> drop rows with any NaN
    skipMiss = False -> pairwise computation
    fun: np.cov (covariance) or np.corrcoef (correlation)
    """
    if skipMiss:
        # skip the row if NaN exists
        clean = x[~np.isnan(x).any(axis=1)]
        return fun(clean, rowvar=False)
    else:
        # pairwise
        n = x.shape[1]
        out = np.full((n, n), np.nan)
        for i in range(n):
            for j in range(n):
                xi = x[:, i]
                xj = x[:, j]
                mask = ~np.isnan(xi) & ~np.isnan(xj)
                if mask.sum() > 1:
                    if fun == np.cov:
                        out[i, j] = np.cov(xi[mask], xj[mask])[0, 1]
                    elif fun == np.corrcoef:
                        out[i, j] = np.corrcoef(xi[mask], xj[mask])[0, 1]
        return out

x = pd.read_csv("test1.csv").values

# 1.1 Skip Missing - Covariance
cout = missing_cov(x, skipMiss=True, fun=np.cov)
pd.DataFrame(cout).to_csv("medy_testout_1.1.csv", index=False)

# 1.2 Skip Missing - Correlation
cout = missing_cov(x, skipMiss=True, fun=np.corrcoef)
pd.DataFrame(cout).to_csv("medy_testout_1.2.csv", index=False)

# 1.3 Pairwise - Covariance
cout = missing_cov(x, skipMiss=False, fun=np.cov)
pd.DataFrame(cout).to_csv("medy_testout_1.3.csv", index=False)

# 1.4 Pairwise - Correlation
cout = missing_cov(x, skipMiss=False, fun=np.corrcoef)
pd.DataFrame(cout).to_csv("medy_testout_1.4.csv", index=False)


In [12]:
X = pd.read_csv("test2.csv").values

cols = [f"x{i+1}" for i in range(X.shape[1])]

def ew_covar(X, lam: float):
    """
    Exponentially-weighted covariance using *batch weighted* formula.

    Weights: w_t ∝ lam^(T-1-t), normalized to sum to 1.
    Mean:    mu = sum_t w_t * x_t   (contemporaneous weighted mean)
    Cov:     Σ  = sum_t w_t * (x_t - mu)(x_t - mu)^T
    """
    X = np.asarray(X, dtype=float)
    T, p = X.shape

    # weights: newest observation gets weight 1 (before normalization)
    # t = 0..T-1 corresponds to oldest..newest
    w = lam ** np.arange(T-1, -1, -1)   # [lam^(T-1), ..., lam^0]
    w = w / w.sum()                     # normalize to sum=1

    # weighted mean
    mu = np.average(X, axis=0, weights=w)

    # weighted, centered outer products
    Xc = X - mu
    # Σ = Xc^T diag(w) Xc
    S = (Xc * w[:, None]).T @ Xc
    return S

# 2.1 EW Covariance (lambda = 0.97)
C_97 = ew_covar(X, 0.97)
# cols = [f"x{i+1}" for i in range(C_97.shape[0])]
pd.DataFrame(C_97, columns=cols).to_csv("medy_testout_2.1.csv", index=False)

# 2.2 EW Correlation (lambda = 0.94)
C_94 = ew_covar(X, 0.94)
sd_inv = 1.0 / np.sqrt(np.diag(C_94))
Dinv = np.diag(sd_inv)
R_94 = Dinv @ C_94 @ Dinv
pd.DataFrame(R_94, columns=cols).to_csv("medy_testout_2.2.csv", index=False)

# 2.3 Var(0.97) + Corr(0.94)
sd_97 = np.sqrt(np.diag(C_97))
D97 = np.diag(sd_97)
Sigma_mix = D97 @ R_94 @ D97
pd.DataFrame(Sigma_mix, columns=cols).to_csv("medy_testout_2.3.csv", index=False)

In [13]:
from pathlib import Path


def symmetrize(A):
    return (A + A.T) / 2.0

def eig_clip_psd(A, eps=0.0):
    """ symmetric matrix A have eigenvalues, clip them to be at least eps，and re-construct the matrix. """
    A = symmetrize(A)
    vals, vecs = np.linalg.eigh(A)
    vals_clipped = np.maximum(vals, eps)
    return (vecs * vals_clipped) @ vecs.T  # vecs @ diag(vals_clipped) @ vecs.T

def near_psd_cov(S, eps=0.0):
    """ Near-PSD for a covariance matrix: eigenvalue clipping and symmetrization. """
    S2 = eig_clip_psd(S, eps=eps)

    #    T = diag( sqrt(diag(S)) / sqrt(diag(S_psd)) ),   S_new = T * S_psd * T
    d_old = np.sqrt(np.clip(np.diag(S),     1e-16, None))
    d_new = np.sqrt(np.clip(np.diag(S2), 1e-16, None))
    T = np.diag(d_old / d_new)

    S = T @ S2 @ T
    return symmetrize(S)


def near_psd_corr(R, eps=0.0):
    """ Near-PSD for a correlation matrix:
    eigenvalue clipping, then renormalize the diagonal to 1
    via a similarity scaling; finally enforce symmetry and unit diagonal."""
    R2 = eig_clip_psd(R, eps=eps)
    d = np.sqrt(np.clip(np.diag(R2), 1e-16, None))
    Dinv = np.diag(1.0 / d)
    R3 = Dinv @ R2 @ Dinv
    np.fill_diagonal(R3, 1.0)
    return symmetrize(R3)



def higham_corr(A, max_iter=100, tol=1e-10):
    A = symmetrize(np.array(A, dtype=float))
    Y = A.copy()
    deltaS = np.zeros_like(A)

    for a in range(max_iter):
        R = Y - deltaS
        R_psd = eig_clip_psd(R, eps=0.0)
        deltaS = R_psd - R
        Y = R_psd.copy()
        np.fill_diagonal(Y, 1.0)

        if np.linalg.norm(Y - R_psd) < tol:   # 2nd term: ord:'fro'
            break

    Y = symmetrize(Y)
    np.fill_diagonal(Y, 1.0)
    return Y

def higham_cov(S, max_iter=100, tol=1e-10):

    S = symmetrize(np.array(S, dtype=float))
    std = np.sqrt(np.clip(np.diag(S), 1e-16, None))
    D = np.diag(std)
    Dinv = np.diag(1.0 / std)

    R = Dinv @ S @ Dinv
    Rn = higham_corr(R, max_iter=max_iter, tol=tol)
    Sn = D @ Rn @ D
    return symmetrize(Sn)

# Path("data").mkdir(exist_ok=True)

# 3.1 near_psd covariance
cin = pd.read_csv("medy_testout_1.3.csv").values
cout = near_psd_cov(cin, eps=0.0)
cols = [f"x{i+1}" for i in range(cout.shape[0])]
pd.DataFrame(cout, columns=cols, index=cols).to_csv("medy_testout_3.1.csv", index=False)

# 3.2 near_psd Correlation
cin = pd.read_csv("medy_testout_1.4.csv").values
cout = near_psd_corr(cin, eps=0.0)
pd.DataFrame(cout, columns=cols, index=cols).to_csv("medy_testout_3.2.csv", index=False)

# 3.3 Higham covariance
cin = pd.read_csv("medy_testout_1.3.csv").values
cout = higham_cov(cin, max_iter=200, tol=1e-10)
pd.DataFrame(cout, columns=cols, index=cols).to_csv("medy_testout_3.3.csv", index=False)

# 3.4 Higham Correlation
cin = pd.read_csv("medy_testout_1.4.csv").values
cout = higham_corr(cin, max_iter=200, tol=1e-10)
pd.DataFrame(cout, columns=cols, index=cols).to_csv("medy_testout_3.4.csv", index=False)


In [14]:
import numpy as np
import pandas as pd

def cholesky_psd_jitter(A, lower=True, jitter0=1e-20, max_tries=8):
    """
    Cholesky factorization for PSD matrices by adding a small diagonal jitter
    if the standard Cholesky fails. Returns a lower-triangular factor L s.t.
    L @ L.T ≈ A  (if lower=True).
    """
    A = np.array(A, dtype=float)
    A = (A + A.T) / 2.0  # enforce symmetry
    jitter = 0.0
    for k in range(max_tries + 1):
        try:
            L = np.linalg.cholesky(A + jitter * np.eye(A.shape[0]))
            return L if lower else L.T
        except np.linalg.LinAlgError:
            if k == 0:
                jitter = jitter0
            else:
                jitter *= 10.0  # grow jitter
    # last resort: raise
    raise np.linalg.LinAlgError("Cholesky failed even with jitter.")

cin = pd.read_csv("medy_testout_3.1.csv").values
L = cholesky_psd_jitter(cin, lower=True)
cols = [f"x{i+1}" for i in range(L.shape[0])]
pd.DataFrame(L, columns=cols, index=cols).to_csv("medy_testout_4.1.csv", index=False)


In [15]:
import numpy as np
import pandas as pd
from pathlib import Path

def return_calculate(df: pd.DataFrame, method: str = "ARITH", date_column: str = "Date") -> pd.DataFrame:
    """
    Calculate arithmetic returns or log returns column-wise.
    Assumes `date_column` holds the date; all other columns are price series.
    The first row's returns are NaN (no prior price).
    """
    # Make a copy and ensure Date is datetime
    out = df.copy()
    if date_column in out.columns:
        out[date_column] = pd.to_datetime(out[date_column])
        out = out.sort_values(date_column).reset_index(drop=True)

    # Identify price columns
    price_cols = [c for c in out.columns if c != date_column]

    # Avoid division by zero for log returns by replacing exact zeros with NaN
    Z = out[price_cols].replace(0.0, np.nan)

    if method.upper() in ("ARITH", "SIMPLE"):
        rets = Z.pct_change()  # (P_t / P_{t-1}) - 1
    elif method.upper() == "LOG":
        rets = np.log(Z / Z.shift(1))  # log(P_t/P_{t-1})
    else:
        raise ValueError("method must be 'ARITH' (or 'SIMPLE') or 'LOG'")

    # Build result DataFrame: keep the Date column and the returns
    rout = pd.concat([out[[date_column]], rets], axis=1)
    return rout

# --- 6.1 Arithmetic returns ---
prices = pd.read_csv("test6.csv")
rout = return_calculate(prices, method="ARITH", date_column="Date")
Path("data").mkdir(exist_ok=True)
rout.to_csv("medy_testout_6.1.csv", index=False)

# --- 6.2 Log returns ---
prices = pd.read_csv("test6.csv")
rout = return_calculate(prices, method="LOG", date_column="Date")
rout.to_csv("medy_testout_6.2.csv", index=False)
